# Pinball — run on Colab

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/DavidvanBruggen/pinball/blob/main/pinball_collab_run.ipynb)

Hierarchical graph transformer for long-context sequence modeling.

**Before running:** pick a GPU runtime — *Runtime → Change runtime type → GPU* (a free T4 works). Then *Runtime → Run all*. The cells clone the repo, install Pinball, fetch WikiText-103, and start training.


In [1]:
!git clone https://github.com/DavidvanBruggen/pinball.git
%cd /content/pinball
%pip install -e .
#%pip install -e ".[flash]" # if running compatible CUDA / GPU
%pip install -q datasets

Cloning into 'pinball'...
remote: Enumerating objects: 113, done.
remote: Counting objects: 100% (113/113), done.
remote: Compressing objects: 100% (79/79), done.
remote: Total 113 (delta 47), reused 87 (delta 27), pack-reused 0 (from 0)
Receiving objects: 100% (113/113), 326.50 KiB | 1.33 MiB/s, done.
Resolving deltas: 100% (47/47), done.
/content/pinball
Obtaining file:///content/pinball
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.4/64.4 kB 5.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.9/72.9 kB 8.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 300.8/300.8 kB 23.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 67.1 MB/s eta 0:00:00
  Building editable for pinball (pyproject.toml) ... done
  Created wheel for pin

In [2]:
from datasets import load_dataset
from pathlib import Path

dataset_name = "Salesforce/wikitext"
config = "wikitext-103-raw-v1"

out_dir = Path("/content/pinball/data/wikitext-103-raw-v1")
out_dir.mkdir(parents=True, exist_ok=True)

for split in ["train"]:#, "validation", "test"]:
    print(f"Loading {split}...")
    ds = load_dataset(dataset_name, config, split=split)

    out_file = out_dir / f"{split}.txt"
    print(f"Writing {out_file}...")

    with out_file.open("w", encoding="utf-8") as f:
        for row in ds:
            text = row["text"]
            # Preserve blank lines, but avoid accidental double newlines
            f.write(text.rstrip("\n") + "\n")

    print(f"Done: {out_file}")

Loading train...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

wikitext-103-raw-v1/test-00000-of-00001.(…):   0%|          | 0.00/733k [00:00<?, ?B/s]

wikitext-103-raw-v1/train-00000-of-00002(…):   0%|          | 0.00/157M [00:00<?, ?B/s]

...

wikitext-103-raw-v1/validation-00000-of-(…):   0%|          | 0.00/657k [00:00<?, ?B/s]

Generating test split:   0%|          | 0/4358 [00:00<?, ? examples/s]

Generating train split:   0%|          | 0/1801350 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/3760 [00:00<?, ? examples/s]

Writing /content/pinball/data/wikitext-103-raw-v1/train.txt...
Done: /content/pinball/data/wikitext-103-raw-v1/train.txt


Run the model with modest settings. It will tokenize if no dataset ".pt" file exists.

In [ ]:
!pinball-train --config configs/pinball_wikitext.yaml \
  --block_size 1024 --batch_size 4 --grad-accum 8 \
  --attn-backend sdpa \
  --text-file /content/pinball/data/wikitext-103-raw-v1/train.txt

2026-06-17 15:21:00,426 INFO HTTP Request: HEAD https://huggingface.co/gpt2/resolve/main/config.json "HTTP/1.1 200 OK"
2026-06-17 15:21:00,427 WARNING Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.
2026-06-17 15:21:00,516 INFO HTTP Request: GET https://huggingface.co/gpt2/resolve/main/config.json "HTTP/1.1 200 OK"
config.json: 100% 665/665 [00:00<00:00, 2.93MB/s]
2026-06-17 15:21:00,618 INFO HTTP Request: HEAD https://huggingface.co/gpt2/resolve/main/tokenizer_config.json "HTTP/1.1 200 OK"
2026-06-17 15:21:00,705 INFO HTTP Request: GET https://huggingface.co/gpt2/resolve/main/tokenizer_config.json "HTTP/1.1 200 OK"
tokenizer_config.json: 100% 26.0/26.0 [00:00<00:00, 124kB/s]
2026-06-17 15:21:00,790 INFO HTTP Request: GET https://huggingface.co/api/models/gpt2/tree/main/additional_chat_templates?recursive=false&expand=false "HTTP/1.1 307 Temporary Redirect"
2026-06-17 15:21:00,876 INFO HTTP Reques